# 24 - What the model says, before it has been allowed to say anything

**Purpose.** Read `results/model_constants.json` back through `model.py` and explain what the
assembled model predicts, what in it is defensible today, and what is still a placeholder. For
someone deciding what to shoot next, and what to measure next.

This notebook measures nothing and writes nothing. Every number below is read from `results/` or
computed from those numbers by `astropix.model`. If anything here disagrees with `results/`,
`results/` is right and this notebook is the bug.

**What it is not for.** It does not rank the exposure ladder. That is the definition-of-done test
and it needs `eta_comb` on registered lights, which contract 3 has not measured yet. What follows
is the shape of the answer, not the answer.

In [ ]:
import json
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from astropix import model as M

plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
K = json.loads((RESULTS / "model_constants.json").read_text())
c = M.Constants.load(RESULTS / "model_constants.json")

G, RC = c["system_gain"], c["read_noise_counts"]
F_SKY, DARK, T_DEAD = c.pick("F_sky", "G1"), c["dark_current_bound"], c["t_dead"]
A = F_SKY + DARK                       # the flux that shot noise is made of
HCG, PED, LIN = c["hcg_threshold_gain"], c["pedestal_fit"], c["linear_to_at_least"]
LADDER_GAIN = 252

nulls = [k for k, v in K.items() if v["value"] is None]
print(f"{len(K)} constants, {len(K) - len(nulls)} measured, {len(nulls)} null with a reason")
print(f"  null: {', '.join(nulls)}\n")
print("where each measured term came from:")
for name in ("system_gain", "read_noise_counts", "F_sky", "t_dead", "dark_current_bound",
             "eta_comb", "linear_to_at_least"):
    v = K[name]
    print(f"  {name:20s} {v['notebook']:24s} {v['measured_on']}  "
          f"{v['source_frames']:5d} frames")

## 1. Read noise is a step function of gain, and the step is the whole story

`R` in *counts* rises with gain, which looks like bad news and is not: counts are not electrons,
and what the model needs is electrons. Multiply by `g` and the picture inverts - read noise in
electrons falls with gain, as it should, because that is what an amplifier in front of a converter
is for.

**The fall is not smooth.** Across the HCG threshold at gain 200 the readout changes conversion
branch and `R` drops from 3.45 to 0.99 electrons in ten gain units - a factor of 3.5 across a step
the control treats as one notch. Everything above it is a long flat tail: 0.99 at gain 200 against
0.67 at gain 450, for nine times the amplification.

That step is why `model.g_at` interpolates between the measured rungs rather than using the fitted
gain law. A single line through the whole domain smears a discontinuity the sensor really has, and
it is the one place in the gain axis where a few units change anything.

In [ ]:
gains = np.array(sorted(int(k) for k in RC))
r_counts = np.array([RC[str(g)] for g in gains])
r_e = np.array([M.read_noise_e(g, RC, G) for g in gains])
g_e = np.array([M.g_at(g, G) for g in gains])

print("gain      g      R counts   R electrons")
for gain in (0, 50, 100, 190, 200, 250, 300, 450):
    print(f"{gain:4d}  {M.g_at(gain, G):7.4f}  {M._interp_table(RC, gain):8.3f}   "
          f"{M.read_noise_e(gain, RC, G):8.3f}")

lo, hi = M.read_noise_e(190, RC, G), M.read_noise_e(200, RC, G)
print(f"\nacross the HCG threshold at gain {HCG}: {lo:.3f} -> {hi:.3f} e-, "
      f"a factor of {lo / hi:.2f} in ten gain units")
print(f"from there to the top of the domain: {hi:.3f} -> {M.read_noise_e(450, RC, G):.3f} e-, "
      f"a factor of {hi / M.read_noise_e(450, RC, G):.2f} in two hundred and fifty")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.6, 2.9))
ax1.plot(gains, r_counts, lw=1.2)
ax1.set(xlabel="gain", ylabel="read noise, ADC counts", title="in counts, it rises")
ax2.plot(gains, r_e, lw=1.2)
ax2.set(xlabel="gain", ylabel="read noise, electrons", title="in electrons, it steps down")
for ax in (ax1, ax2):
    ax.axvline(HCG, color="0.7", lw=0.8)
    ax.annotate("HCG", (HCG, ax.get_ylim()[1]), fontsize=7, ha="left", va="top", color="0.4")
fig.tight_layout()

## 2. There is no best sub length, and that changes the question

Differentiate the SNR at fixed wall clock and the stationary condition reduces to
`t*(R^2 + A*t_dead) + 2*R^2*t_dead = 0`, whose left side is strictly positive for every positive
`t`. **SNR climbs with sub length forever and flattens; it never turns over.** `eta_comb` pushes
the same way, since longer subs mean fewer of them and this rig's combination efficiency falls
with stack size.

So `model.py` has no `optimal_t`, and a function with that name would have had to invent a
stopping rule to return anything. What it has instead is `efficiency` - how much of the asymptote
a given `t` reaches - and it factors into two readable pieces:

```
eff(t) = sqrt(  t/(t + t_dead)   x   A*t/(A*t + R^2)  )
              exposing fraction     sky-dominated fraction
```

The left factor is the fraction of the night that is actually collecting. The right is the
fraction of the variance that is sky and thermal rather than read. **On this rig the left factor
is the smaller one at every gain above the HCG threshold**, which is not how the sub-length
question is usually framed.

In [ ]:
def eff(t, gain):
    return M.efficiency(t, f_sky=F_SKY, dark=DARK, t_dead=T_DEAD,
                        read_e=M.read_noise_e(gain, RC, G))


no_read = M.efficiency(60, f_sky=F_SKY, dark=DARK, read_e=0.0, t_dead=T_DEAD)
print(f"at a 60 s sub, dead time alone costs {100 * (1 - no_read):.2f}% of the asymptote, "
      f"at every gain.\nwhat read noise adds on top:")
for gain in (0, 50, 100, 190, 200, 300, 450):
    print(f"  gain {gain:3d}: {100 * (1 - eff(60, gain) / no_read):6.3f}%")

fig, ax = plt.subplots(figsize=(5.4, 3.1))
t = np.logspace(np.log10(5), np.log10(1800), 400)
read_e = M.read_noise_e(LADDER_GAIN, RC, G)
ax.plot(t, np.sqrt(t / (t + T_DEAD)), lw=1.1, ls="--", label=f"exposing, t_dead {T_DEAD:.1f} s")
ax.plot(t, np.sqrt(A * t / (A * t + read_e ** 2)), lw=1.1, ls=":",
        label=f"sky-dominated, R {read_e:.2f} e-")
ax.plot(t, [eff(x, LADDER_GAIN) for x in t], lw=1.6, color="k", label="efficiency, the product")
for target in (0.90, 0.95):
    ax.axhline(target, color="0.85", lw=0.8)
ax.set(xscale="log", ylim=(0.4, 1.02), xlabel="sub length, s",
       ylabel="fraction of the asymptotic SNR",
       title=f"the two factors, at gain {LADDER_GAIN}")
ax.legend(fontsize=7, loc="lower right")
fig.tight_layout()

for target in (0.90, 0.95, 0.99):
    t_need = M.t_for_efficiency(target, f_sky=F_SKY, dark=DARK, read_e=read_e, t_dead=T_DEAD)
    print(f"{target:.0%} of the asymptote needs t = {t_need:7.1f} s")

### Read noise is already beaten at the shortest rung on the ladder

At gain 252 a 15 s sub is **97.4% sky-dominated**. Read noise costs 1.3% of the
efficiency there; dead time costs 43% of it, because 67.7% of the night collects nothing. MISSION's own text says the whole
sub-exposure question lives in `R^2/t`; on this rig, at this sky, above the HCG threshold, it
does not. **It lives in `t_dead`**, and `t_dead` is 31.5 s because this night dithered after every
frame.

That is a statement about the observing pattern, not about the camera - which makes it the more
actionable of the two. Read noise is fixed by the sensor. Dithering every second frame instead of
every frame roughly halves the overhead, and it is a setting in the ASIAIR.

In [ ]:
print(f"at gain {LADDER_GAIN}, over a six-hour night:\n")
print("    t   exposing  sky-dom   eff     subs   subs if dithering every 2nd frame")
for t in (15, 30, 60, 120, 240, 480):
    half = M.subs_in(6 * 3600, t, T_DEAD / 2)
    print(f"{t:5d}    {t / (t + T_DEAD):.4f}   {A * t / (A * t + read_e ** 2):.4f}  "
          f"{eff(t, LADDER_GAIN):.4f}  {M.subs_in(6 * 3600, t, T_DEAD):5d}   {half:5d}"
          f"   (+{100 * (half / M.subs_in(6 * 3600, t, T_DEAD) - 1):.0f}%)")

gain_hi = M.efficiency(60, f_sky=F_SKY, dark=DARK, read_e=read_e, t_dead=T_DEAD / 2)
print(f"\nhalving the overhead lifts a 60 s sub from {eff(60, LADDER_GAIN):.4f} to "
      f"{gain_hi:.4f} of the asymptote -- worth more than any gain change above HCG")

## 3. The other constraint, and why it points at one gain

The star-colour constraint is `pedestal + (F_star + F_sky + D)*t/g <= ceiling(gain)`, so the
longest unclipped sub scales with `g`. Gain buys read noise and spends star colour, and those are
the two arms of MISSION's Pareto curve.

Above the HCG threshold the first arm is **flat**: efficiency at 60 s moves from 0.8065 at gain
200 to 0.8084 at gain 450, a gain of 0.24% for nine times the amplification. The second arm is
not flat at all - it falls with `g`, which is to say by a factor of nineteen over the same range.

**So gain 200 dominates everything above it** - not a trade, a dominance. There is no point on the
curve between 200 and 450 that is better at anything. Below 200 there is a real trade, because
read noise is still costing 4.8% at gain 190 and 13.3% at gain 0, and longer unclipped subs are
available to pay for it.

The star below collects 400 e-/s in its brightest pixel, which is a choice of star and not a
measurement; what transfers is the ratio between rows, since `t_max` is linear in the star's flux.

In [ ]:
STAR = 400.0                           # e-/s in the peak pixel; a choice, not a constant

print(f"a star core at {STAR:.0f} e-/s, with the sky and dark underneath it:\n")
print("gain   pedestal   ceiling   t_max unclipped   eff at 60 s")
for gain in (50, 100, 200):
    ped = M.pedestal_counts(gain, PED, HCG)
    t_max = M.t_max_colour(STAR, f_sky=F_SKY, dark=DARK, pedestal=ped,
                           ceiling=LIN[str(gain)], g=M.g_at(gain, G))
    print(f"{gain:4d}   {ped:8.2f}   {LIN[str(gain)]:7.1f}   {t_max:12.1f} s   "
          f"{eff(60, gain):11.4f}")

print(f"\nthe ceiling is not measured above gain 200, so gain {LADDER_GAIN} and gain 450 "
      f"cannot be put in this table.")
print(f"scaling t_max by g alone, gain 450 would reach roughly "
      f"{M.t_max_colour(STAR, f_sky=F_SKY, dark=DARK, pedestal=M.pedestal_counts(450, PED, HCG), ceiling=LIN['200'], g=M.g_at(450, G)):.2f} s "
      f"-- an estimate that borrows gain 200's ceiling and is shown to give the magnitude, "
      f"not as a number to use")

fig, ax = plt.subplots(figsize=(5.4, 3.1))
grid = np.arange(0, 451, 10)
ax.plot(grid, [eff(60, x) / eff(60, 450) for x in grid], lw=1.4, label="efficiency at 60 s")
ax.plot(grid, [M.g_at(x, G) / M.g_at(200, G) for x in grid], lw=1.4,
        label="t_max unclipped (proportional to g)")
ax.axvline(HCG, color="0.7", lw=0.8)
ax.set(yscale="log", xlabel="gain", ylabel="relative to its own value at the reference gain",
       title="one arm flattens at HCG, the other keeps falling")
ax.legend(fontsize=7)
fig.tight_layout()

## 4. What is defensible today, and what is a placeholder

**Defensible.** `g`, `R`, `F_sky`, `t_dead`, the pedestal and the linearity ceiling are measured
with provenance, and everything in sections 1 to 3 follows from them by arithmetic this notebook
does not hide. The efficiency curve, the HCG step, and the dominance argument for gain 200 are
predictions this project can trace back to its own frames.

**A placeholder.** `eta_comb`. The one in this file was measured on **bias stacks** and stalls
against a fixed-pattern floor, falling to 0.536 by N=128. Its own note calls it an upper bound on
the real loss. It is the term that decides whether many short subs beat few long ones, and it is
the term the model is least entitled to quote.

**Out of reach entirely.** Any ranking of the NGC 7000 exposure ladder. A six-hour night of 15 s
subs is N=464, past the end of the measured `eta_comb` ladder, and `model.eta_at` refuses to
extrapolate - which is the gate working rather than a gap in it.

In [ ]:
n_max = max(int(k) for k in c["eta_comb"])
print(f"eta_comb measured to N={n_max}, on {K['eta_comb']['notebook']}'s bias stacks:")
for n in sorted(int(k) for k in c["eta_comb"]):
    print(f"  N={n:4d}  eta {c['eta_comb'][str(n)]:.4f}   "
          f"costs {100 * (1 - c['eta_comb'][str(n)]):5.2f}%")

print(f"\nsubs in a six-hour night at gain {LADDER_GAIN}, and whether eta_comb covers them:")
for t in (15, 30, 60, 120, 240, 480):
    n = M.subs_in(6 * 3600, t, T_DEAD)
    try:
        verdict = f"eta {M.eta_at(n, c['eta_comb']):.4f}"
    except ValueError:
        verdict = f"REFUSED -- past the measured ladder (N={n_max})"
    print(f"  {t:4d} s -> N={n:4d}   {verdict}")

## 5. What to do next, in order

1. **Contract 3: `eta_comb` on registered lights.** The only term blocking a ranking, and the one
   whose current value is least trustworthy. It also settles MISSION's surviving assumption -
   whether a noise term exists that fails to scale with `t` - because that is exactly what a
   stalling `eta_comb` is made of. The ladder must reach N of several hundred, or the short rungs
   stay unrankable.
2. **A linearity block at gain 252.** A short bench session. Without it the star-colour half of
   the model cannot be evaluated on the one dataset the definition of done is written against.
3. **The dithering decision.** Halving `t_dead` is worth more at 60 s than every gain change above
   the HCG threshold put together, and it costs nothing but a setting. Whether dithering every
   second frame still rejects well enough is a question for contract 3's real stacks, which makes
   it the same experiment.

**Not next: a gain sweep.** Above the HCG threshold gain changes efficiency by a quarter of a
percent, and the model already says so from measured constants. A sweep there would pin down no
term. Below the threshold the trade is real, but nothing recommends observing there.